# 阶段 2 · 步骤 3：批量生成图数据 → PyG `Data` 对象

这是数据生成的核心步骤。目标：写三个函数，把「随机图 + Dijkstra 标签」变成 GNN 能吃的训练样本。

**三个函数的分工**：

| 函数 | 干什么 |
|------|--------|
| `generate_graph()` | 随机生成一个带权连通图 |
| `get_shortest_path()` | 用 Dijkstra 算出真实最短路径（标签来源） |
| `convert_to_pyg()` | 把图转成 PyG 的 `Data` 对象 |

每个 cell 先看讲解，再补全 TODO，然后运行验证。逐 cell 往下走。

In [ ]:
import random

import networkx as nx
import torch
from torch_geometric.data import Data

print('imports OK')

## 1. 随机图生成 `generate_graph()`

用 **Erdős-Rényi 模型**：N 个节点，每对节点之间以概率 p 连一条边。

- 节点数随机在 `[min_nodes, max_nodes]` 之间
- 连边概率 p 随机在 `[p_min, p_max]` 之间 —— 让训练数据有多样性
- 每条边加一个随机权重 `[0.1, 1.0]`
- 用 `random.Random(seed)` 保证可复现（上次练习你已经用过）

In [ ]:
def generate_graph(min_nodes=10, max_nodes=16, p_min=0.25, p_max=0.4, rng=None):
    """生成一个带权随机图（Erdos-Renyi 模型）。

    Args:
        min_nodes / max_nodes: 节点数范围
        p_min / p_max: 连边概率范围
        rng: random.Random 实例（可复现）

    Returns:
        networkx.Graph，边带 weight 属性，取值 [0.1, 1.0]
    """
    if rng is None:
        rng = random.Random()

    # TODO 1: 随机决定节点数 n（rng.randint）
    n = None

    # TODO 2: 随机决定连边概率 p（rng.uniform）
    p = None

    # TODO 3: 用 nx.gnp_random_graph(n, p, seed=...) 生成图
    # 提示: seed 用 rng.randint(0, 2**31 - 1)
    G = None

    # TODO 4: 给每条边加随机权重（你在 try_graph.py 写过类似的）
    # 提示: 遍历 G.edges()，用 rng.uniform(0.1, 1.0)

    return G

# 验证：生成一个图看看
G = generate_graph(rng=random.Random(0))
print(f"节点数: {G.number_of_nodes()}, 边数: {G.number_of_edges()}")
print(f"连通: {nx.is_connected(G)}")  # 注意：可能不连通！这是下一步要筛掉的

## 2. 筛选合格样本

不是每个随机图都能当训练样本，要满足三个条件：

1. **连通**——不连通的话某些点对之间没有路径
2. **源点 ≠ 目标点**——自己到自己的路径是空的，没有学习价值
3. **最短路径唯一**——如果两条路径并列最短，标签就有歧义（模型学哪条？）
4. **路径长度 ≥ 3 个节点**——只有 2 个节点（s 直连 t）的样本太平凡

检查「路径唯一」的方法：`nx.all_shortest_paths()` 返回所有并列最短路径，数量为 1 才合格。

In [ ]:
def get_shortest_path(G, source, target):
    """用 Dijkstra 计算加权最短路径（try_dijkstra.py 里你已经用过）。"""
    # TODO 5: 一行代码，返回节点列表
    return None

# 验证：反复尝试直到找到一个合格样本，统计尝试次数
rng = random.Random(0)
attempts = 0
while True:
    attempts += 1
    G = generate_graph(rng=rng)
    if not nx.is_connected(G):
        continue  # 条件 1
    source = rng.randrange(G.number_of_nodes())
    target = rng.randrange(G.number_of_nodes())
    if source == target:
        continue  # 条件 2
    path = get_shortest_path(G, source, target)
    if len(path) < 3:
        continue  # 条件 4
    if len(list(nx.all_shortest_paths(G, source, target, weight='weight'))) != 1:
        continue  # 条件 3：路径不唯一
    break

print(f"尝试 {attempts} 次找到合格样本")
print(f"路径: {path}")
print(f"路径长度: {len(path)} 个节点")

## 3. 转成 PyG `Data` —— 本步骤的核心

回忆三种数据的分工（阶段 1 讲过）：

| 数据 | 形状 | 装什么 |
|------|------|--------|
| `x` | `[N, F]` | 节点特征 |
| `edge_index` | `[2, 2E]` | 边连接（无向边存两个方向！） |
| `edge_attr` | `[2E, D]` | 边特征（权重） |
| `y` | `[2E]` | 每条有向边的标签 |

**节点特征设计（5 维，不用 Dijkstra 结果，只用无权 BFS 跳数）**：

```
[是否为源点, 是否为目标点, 到源点的跳数, 到目标点的跳数, 度]
```

都归一化到 0~1。为什么用 BFS 跳数？——它是「拓扑位置」的线索：最短路径上的节点大致「顺着源点到目标点的方向排列」。

**标签**：无向边 (u,v) 存成两条有向边 (u→v) 和 (v→u)，两条的标签相同（都是 1.0 或都是 0.0）。

In [ ]:
def convert_to_pyg(G, source, target, path):
    """将 networkx 图转换为 PyG 的 Data 对象。

    Args:
        G: networkx 图
        source / target: 源点 / 目标点
        path: 真实最短路径（节点列表）

    Returns:
        torch_geometric.data.Data
    """
    n = G.number_of_nodes()

    # TODO 6: 算每个节点到 source 和 target 的 BFS 跳数
    # 提示: nx.single_source_shortest_path_length(G, source) —— try_dijkstra.py 的 TODO 4 你写过
    hop_s = None
    hop_t = None

    # TODO 7: 构建节点特征 x，形状 [n, 5]
    # 先 torch.zeros(n, 5)，再逐节点填 5 个值（都除以 max(n-1, 1) 归一化）
    x = None

    # TODO 8: 把路径转成无向边集合
    # 提示: {frozenset((a, b)) for a, b in zip(path[:-1], path[1:])}
    path_edges = None

    # TODO 9: 遍历 G.edges(data=True)，对每条无向边生成两个方向：
    #   edge_index 加 [a, b]；edge_attr 加 [weight, weight/w_max]；y 加 0.0 或 1.0
    # 提示: w_max 是所有边权中的最大值（用于归一化）
    edge_index, edge_attr, y = [], [], []

    data = Data(
        x=torch.tensor(x, dtype=torch.float),
        edge_index=torch.tensor(edge_index, dtype=torch.long).t().contiguous(),
        edge_attr=torch.tensor(edge_attr, dtype=torch.float),
        y=torch.tensor(y, dtype=torch.float),
        source=source,
        target=target,
        path_nodes=list(path),  # 存起来方便以后验证
    )
    return data

data = convert_to_pyg(G, source, target, path)
print(data)  # PyG 自带的打印，能看到各字段形状

## 4. 检查 `Data` 对象

运行下面的 cell，逐项核对：

- `x` 形状应为 `[N, 5]`
- `edge_index` 形状应为 `[2, 2E]`
- `y` 中 1 的个数应等于 `路径节点数 - 1`（路径的边数）
- `x[source, 0] == 1`，`x[target, 1] == 1`

In [ ]:
print(f"节点特征 x:        {list(data.x.shape)}  (期望 [N, 5])")
print(f"边索引 edge_index: {list(data.edge_index.shape)}  (期望 [2, 2E])")
print(f"边特征 edge_attr:  {list(data.edge_attr.shape)}  (期望 [2E, 2])")
print(f"标签 y:            {list(data.y.shape)}  (期望 [2E])")
print()
print(f"正标签数 (y==1): {int(data.y.sum())}  (期望 {len(path) - 1}，即路径边数 × 2 方向 / 2 ... 想想为什么)")
print(f"x[source] = {data.x[source].tolist()}")
print(f"x[target] = {data.x[target].tolist()}")

## 5. 批量生成 `build_dataset()`

把上面所有逻辑串起来：循环「生成 → 筛选 → 转换」，直到攒够指定数量的样本。

In [ ]:
def build_dataset(num_samples, seed=42, verbose=False):
    """批量生成合格样本。

    Args:
        num_samples: 需要的样本数
        seed: 随机种子

    Returns:
        list[Data]
    """
    rng = random.Random(seed)
    samples = []
    attempts = 0
    # TODO 10: 写 while 循环，把第 2 节的筛选逻辑 + convert_to_pyg 串起来
    # 提示: while len(samples) < num_samples: ... 每轮 attempts += 1
    # verbose 时每 50 个打印一次进度
    return samples

train_set = build_dataset(50, seed=0, verbose=True)
print(f"生成 {len(train_set)} 个样本")

## 6. 数据质量检查

养成习惯：生成数据后先看统计分布，再喂模型。检查：
- 节点数 / 边数范围
- 路径长度分布
- **正样本比例**——如果太低（如 < 5%），训练时类别会严重不平衡（阶段 4 会用 `pos_weight` 处理）

In [ ]:
n_nodes = [s.num_nodes for s in train_set]
n_edges = [s.num_edges // 2 for s in train_set]  # 除以 2 还原成无向边数
path_lens = [len(s.path_nodes) for s in train_set]
pos_ratio = sum(int(s.y.sum()) for s in train_set) / (2 * sum(n_edges))

print(f"节点数: {min(n_nodes)} ~ {max(n_nodes)} (平均 {sum(n_nodes)/len(n_nodes):.1f})")
print(f"边数:   {min(n_edges)} ~ {max(n_edges)} (平均 {sum(n_edges)/len(n_edges):.1f})")
print(f"路径长: {min(path_lens)} ~ {max(path_lens)} (平均 {sum(path_lens)/len(path_lens):.1f})")
print(f"正样本边比例: {pos_ratio:.3f}")

## 7. 保存 + 可视化抽查

用 `torch.save` 把数据集存成 `.pt` 文件；再画几个样本肉眼检查标签是否正确（绿线 = 真实最短路径）。

In [ ]:
import os

import matplotlib.pyplot as plt

os.makedirs('data', exist_ok=True)
torch.save({'train': train_set}, 'data/shortest_path_dataset.pt')
print('已保存 data/shortest_path_dataset.pt')

# 可视化 2 个样本：红线 = 真实最短路径
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for i, ax in enumerate(axes):
    s = train_set[i]
    G2 = nx.Graph()
    G2.add_nodes_from(range(s.num_nodes))
    ei = s.edge_index.numpy()
    ea = s.edge_attr.numpy()
    seen = set()
    for k in range(ei.shape[1]):
        u, v = int(ei[0, k]), int(ei[1, k])
        if (min(u,v), max(u,v)) in seen:
            continue
        seen.add((min(u,v), max(u,v)))
        G2.add_edge(u, v, weight=round(float(ea[k, 0]), 2))
    pos = nx.spring_layout(G2, seed=42)
    path_edges = list(zip(s.path_nodes[:-1], s.path_nodes[1:]))
    colors = ['red' if (u, v) in path_edges or (v, u) in path_edges else 'gray'
              for u, v in G2.edges()]
    nx.draw(G2, pos, ax=ax, with_labels=True, node_color='skyblue',
            edge_color=colors, width=2)
    nx.draw_networkx_edge_labels(G2, pos, edge_labels=nx.get_edge_attributes(G2, 'weight'), ax=ax)
    ax.set_title(f"sample {i}: {s.source} -> {s.target}")
plt.savefig('data_check.png', dpi=150, bbox_inches='tight')
plt.show()
print('已保存 data_check.png —— 红线是否真的是权重和最小的路径？')
print('TODO 11（口头验证）: 挑一个样本，手动数一数红线权重和，与某条非红路径比较')

## 检查点（阶段 2 完成标准）

全部答对就进入阶段 3（模型实现）：

1. `edge_index` 为什么是 `[2, 2E]` 而不是 `[2, E]`？
2. 为什么节点特征里放 BFS 跳数而不放 Dijkstra 距离？（提示：想想模型推理时能不能提前知道答案）
3. 为什么无向边 (u,v) 的两个方向标签相同？
4. `build_dataset` 里为什么要用 `attempts` 计数？如果图很难合格会发生什么？